# Hardware Trojan Detection — Power-Trace Model Training

Trains and evaluates three classifiers on the same stratified train/test split:

- **Random Forest**
- **RBF-kernel SVM**
- **Lightweight 1D-CNN**

Input is standardized once using statistics learned only from the training
partition, avoiding test-data leakage.

**Added in this notebook (beyond the original scripts):**
- Leave-one-Trojan-out cross-validation (trains on some Trojan variants, tests on a held-out unseen one — proves generalization, not memorization)
- F1 score for the CNN (previously accuracy-only, now comparable to RF/SVM)
- Confusion matrices for all three models
- Per-variant breakdown of false negative rate (the metric that matters most for Trojan detection)

## Dataset format
Each row is one FPGA power measurement, with a binary/multi-class label column
(e.g. `label` with values `clean`, `trojan_leak`, `trojan_counter`, `trojan_tiny`).
Use either a fixed-length sequence column such as `power_trace`, or one numeric
CSV column per sample.

```
label,power_trace
clean,"[0.12, 0.13, 0.11, ...]"
trojan_leak,"[0.44, 0.41, 0.48, ...]"
```

**Tip:** keep your label column granular (e.g. `trojan_leak` not just `trojan`)
even if your main classification task is binary — the leave-one-out section
below needs to know which Trojan *variant* each row belongs to.


## 1. Install & Import

In [ ]:
# !pip install -q datasets matplotlib numpy pandas scikit-learn torch

import json
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, ConfusionMatrixDisplay


## 2. Data Loading

Load from CSV or a Hugging Face dataset. Either path returns a plain
`pandas.DataFrame`.

In [ ]:
def load_csv_data(csv_path: str | Path) -> pd.DataFrame:
    """Load a CSV file into a DataFrame."""
    return pd.read_csv(csv_path)


def load_huggingface_data(
    dataset_id: str,
    split: str = "train",
    revision: str | None = None,
) -> pd.DataFrame:
    """Load one tabular dataset split from the Hugging Face Hub."""
    try:
        from datasets import load_dataset
    except ImportError as error:
        raise ImportError(
            "Hugging Face support requires 'datasets'. Install it with: "
            "python3 -m pip install datasets"
        ) from error
    return load_dataset(dataset_id, split=split, revision=revision).to_pandas()


## 3. Feature Preparation

Turns raw rows into a finite numeric feature matrix + label vector, then does
a stratified train/test split with a scaler fit only on the training data.

In [ ]:
def power_trace_features(
    data: pd.DataFrame,
    target_column: str,
    trace_column: str | None = None,
) -> tuple[np.ndarray, pd.Series]:
    """Return a finite feature matrix and its label vector.

    ``trace_column`` should contain one fixed-length numeric list per row. If
    omitted, every numeric column except the target becomes a feature.
    """
    if data.empty:
        raise ValueError("The dataset is empty.")
    if target_column not in data.columns:
        raise ValueError(f"Target column '{target_column}' was not found.")

    target = data[target_column]
    if target.isna().any():
        raise ValueError("The target column contains missing values.")
    if target.nunique() < 2:
        raise ValueError("At least two classes are required for classification.")

    if trace_column is None:
        numeric_features = data.drop(columns=[target_column]).select_dtypes(include="number")
        if numeric_features.empty:
            raise ValueError(
                "No numeric features found. Set trace_column to the power-trace column."
            )
        matrix = numeric_features.to_numpy(dtype=np.float32)
    else:
        if trace_column not in data.columns:
            raise ValueError(f"Trace column '{trace_column}' was not found.")
        try:
            traces = [
                json.loads(trace) if isinstance(trace, str) else trace
                for trace in data[trace_column]
            ]
            matrix = np.asarray(traces, dtype=np.float32)
        except (TypeError, ValueError) as error:
            raise ValueError("Each power trace must contain only numeric samples.") from error
        if matrix.ndim != 2 or matrix.shape[1] == 0:
            raise ValueError("Power traces must have the same non-zero length.")

    if not np.isfinite(matrix).all():
        raise ValueError("Features contain missing or non-finite values.")
    return matrix, target.reset_index(drop=True)


def prepare_model_inputs(
    features: np.ndarray,
    target: pd.Series,
    test_size: float = 0.2,
    random_state: int = 42,
) -> dict[str, object]:
    """Create one stratified split and one fitted scaler for every model."""
    if not 0 < test_size < 1:
        raise ValueError("test_size must be a number between 0 and 1.")

    class_counts = target.value_counts()
    if class_counts.min() < 2:
        raise ValueError("Every class needs at least two samples for a stratified split.")

    x_train, x_test, y_train, y_test = train_test_split(
        features,
        target.to_numpy(),
        test_size=test_size,
        random_state=random_state,
        stratify=target.to_numpy(),
    )
    label_encoder = LabelEncoder().fit(y_train)
    try:
        y_train_encoded = label_encoder.transform(y_train)
        y_test_encoded = label_encoder.transform(y_test)
    except ValueError as error:
        raise ValueError("All classes must be represented in the training split.") from error

    scaler = StandardScaler().fit(x_train)
    return {
        "x_train": scaler.transform(x_train).astype(np.float32),
        "x_test": scaler.transform(x_test).astype(np.float32),
        "y_train": y_train_encoded,
        "y_test": y_test_encoded,
        "scaler": scaler,
        "label_encoder": label_encoder,
    }


def save_pickle(model_bundle: object, output_path: str | Path) -> Path:
    """Persist a fitted model bundle to a pickle file."""
    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    with output.open("wb") as file:
        pickle.dump(model_bundle, file)
    return output


## 4. 1D-CNN (PyTorch)

In [ ]:
def _torch_components():
    try:
        import torch
        from torch import nn
    except ImportError as error:
        raise ImportError(
            "1D-CNN training requires PyTorch. Install it with: "
            "python3 -m pip install torch"
        ) from error
    return torch, nn


def _build_network(class_count: int):
    """Build a compact CNN that accepts one normalized power trace per row."""
    _, nn = _torch_components()

    class PowerTraceCNN(nn.Module):
        def __init__(self) -> None:
            super().__init__()
            self.features = nn.Sequential(
                nn.Conv1d(1, 16, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.Conv1d(16, 32, kernel_size=3, padding=1),
                nn.ReLU(),
                nn.AdaptiveAvgPool1d(1),
            )
            self.classifier = nn.Linear(32, class_count)

        def forward(self, traces):
            return self.classifier(self.features(traces).squeeze(-1))

    return PowerTraceCNN()


def train_1d_cnn(
    x_train: np.ndarray,
    y_train: np.ndarray,
    x_test: np.ndarray,
    y_test: np.ndarray,
    *,
    epochs: int = 20,
    batch_size: int = 32,
    learning_rate: float = 1e-3,
    random_state: int = 42,
    show_epoch_progress: bool = True,
):
    """Train the 1D-CNN. Returns (portable_state_dict, predictions, accuracy)."""
    if epochs < 1 or batch_size < 1 or learning_rate <= 0:
        raise ValueError("epochs, batch_size, and learning_rate must be positive.")

    torch, nn = _torch_components()
    torch.manual_seed(random_state)
    model = _build_network(len(np.unique(y_train)))
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.CrossEntropyLoss()
    training_data = torch.utils.data.TensorDataset(
        torch.tensor(x_train).unsqueeze(1), torch.tensor(y_train, dtype=torch.long)
    )
    generator = torch.Generator().manual_seed(random_state)
    loader = torch.utils.data.DataLoader(
        training_data, batch_size=batch_size, shuffle=True, generator=generator
    )

    model.train()
    for epoch in range(1, epochs + 1):
        for traces, labels in loader:
            optimizer.zero_grad()
            loss = criterion(model(traces), labels)
            loss.backward()
            optimizer.step()
        if show_epoch_progress:
            print(f"CNN training: epoch {epoch}/{epochs} completed")

    model.eval()
    with torch.no_grad():
        predictions = model(torch.tensor(x_test).unsqueeze(1)).argmax(dim=1).numpy()
    accuracy = float((predictions == y_test).mean())
    state_dict = {name: value.detach().cpu() for name, value in model.state_dict().items()}
    artifact = {
        "architecture": "PowerTraceCNN",
        "input_length": int(x_train.shape[1]),
        "class_count": int(len(np.unique(y_train))),
        "state_dict": state_dict,
    }
    return artifact, predictions, accuracy


## 5. Evaluation Helpers

Now computes **accuracy + weighted F1 for all three models**, including the
CNN (the original scripts only reported CNN accuracy). Also adds confusion
matrices so you can see exactly which classes get confused with which is
useful for spotting whether your smallest/stealthiest Trojan variant is the
one driving most of the errors.

In [ ]:
def _metrics(y_true: np.ndarray, predictions: np.ndarray) -> dict[str, float]:
    return {
        "accuracy": float(accuracy_score(y_true, predictions)),
        "f1_weighted": float(f1_score(y_true, predictions, average="weighted", zero_division=0)),
    }


def save_evaluation_bar_chart(metrics: dict[str, dict[str, float]], output_path: str | Path) -> Path:
    """Save a test-set metric comparison chart (accuracy + weighted F1) and return its path."""
    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    model_names = list(metrics)
    display_names = [name.replace("_", " ").title() for name in model_names]
    accuracy = [metrics[name]["accuracy"] for name in model_names]
    f1_scores = [metrics[name]["f1_weighted"] for name in model_names]
    positions = list(range(len(model_names)))
    width = 0.36

    figure, axis = plt.subplots(figsize=(9, 5))
    accuracy_bars = axis.bar(
        [p - width / 2 for p in positions], accuracy, width, label="Accuracy", color="#2a9d8f"
    )
    f1_bars = axis.bar(
        [p + width / 2 for p in positions], f1_scores, width, label="Weighted F1", color="#457b9d"
    )
    for bars in (accuracy_bars, f1_bars):
        for bar in bars:
            axis.annotate(
                f"{bar.get_height():.3f}",
                (bar.get_x() + bar.get_width() / 2, bar.get_height()),
                ha="center", va="bottom", fontsize=9,
                xytext=(0, 3), textcoords="offset points",
            )
    axis.set_title("Test-set model evaluation")
    axis.set_ylabel("Score")
    axis.set_ylim(0, 1.08)
    axis.set_xticks(positions, display_names)
    axis.legend()
    axis.grid(axis="y", alpha=0.25)
    figure.tight_layout()
    figure.savefig(output, dpi=160)
    plt.show()
    return output


def plot_confusion_matrices(
    y_test: np.ndarray,
    predictions: dict[str, np.ndarray],
    label_encoder: LabelEncoder,
    output_path: str | Path,
) -> Path:
    """Save side-by-side confusion matrices for all models."""
    output = Path(output_path)
    output.parent.mkdir(parents=True, exist_ok=True)
    class_names = label_encoder.classes_

    fig, axes = plt.subplots(1, len(predictions), figsize=(6 * len(predictions), 5))
    if len(predictions) == 1:
        axes = [axes]
    for axis, (name, preds) in zip(axes, predictions.items()):
        cm = confusion_matrix(y_test, preds)
        disp = ConfusionMatrixDisplay(cm, display_labels=class_names)
        disp.plot(ax=axis, colorbar=False, xticks_rotation=45)
        axis.set_title(name.replace("_", " ").title())
    fig.tight_layout()
    fig.savefig(output, dpi=160)
    plt.show()
    return output


## 6. Training of All Three Models (standard random split)

This is the baseline evaluation i.e, a stratified random train/test split.
Good for a quick sanity check, but see **Section 7** for the validation
method you should actually report in your final results.

In [ ]:
def train_three_models(
    data: pd.DataFrame,
    target_column: str,
    output_path: str | Path = "artifacts/hardware_trojan_models.pkl",
    *,
    trace_column: str | None = None,
    test_size: float = 0.2,
    random_state: int = 42,
    cnn_epochs: int = 20,
    cnn_batch_size: int = 32,
    cnn_learning_rate: float = 1e-3,
    show_epoch_progress: bool = True,
    evaluation_plot_path: str | Path | None = None,
    confusion_matrix_path: str | Path | None = None,
) -> dict:
    """Train SVM, Random Forest, and 1D-CNN using one shared data split."""
    features, target = power_trace_features(data, target_column, trace_column)
    prepared = prepare_model_inputs(features, target, test_size, random_state)
    x_train, x_test = prepared["x_train"], prepared["x_test"]
    y_train, y_test = prepared["y_train"], prepared["y_test"]

    random_forest = RandomForestClassifier(
        n_estimators=300, random_state=random_state, n_jobs=-1, class_weight="balanced",
    ).fit(x_train, y_train)
    svm = SVC(
        kernel="rbf", C=1.0, gamma="scale", class_weight="balanced",
        probability=True, random_state=random_state,
    ).fit(x_train, y_train)
    cnn_artifact, cnn_preds, cnn_accuracy = train_1d_cnn(
        x_train, y_train, x_test, y_test,
        epochs=cnn_epochs, batch_size=cnn_batch_size, learning_rate=cnn_learning_rate,
        random_state=random_state, show_epoch_progress=show_epoch_progress,
    )

    rf_preds = random_forest.predict(x_test)
    svm_preds = svm.predict(x_test)
    predictions = {"random_forest": rf_preds, "svm": svm_preds, "cnn_1d": cnn_preds}

    model_bundle = {
        "format_version": 2,
        "feature_count": int(features.shape[1]),
        "target_column": target_column,
        "trace_column": trace_column,
        "scaler": prepared["scaler"],
        "label_encoder": prepared["label_encoder"],
        "models": {"random_forest": random_forest, "svm": svm, "cnn_1d": cnn_artifact},
        "metrics": {
            "random_forest": _metrics(y_test, rf_preds),
            "svm": _metrics(y_test, svm_preds),
            "cnn_1d": _metrics(y_test, cnn_preds),
        },
    }
    saved_path = save_pickle(model_bundle, output_path)

    plot_path = Path(evaluation_plot_path) if evaluation_plot_path else saved_path.with_suffix(".evaluation.png")
    save_evaluation_bar_chart(model_bundle["metrics"], plot_path)

    cm_path = Path(confusion_matrix_path) if confusion_matrix_path else saved_path.with_suffix(".confusion.png")
    plot_confusion_matrices(y_test, predictions, prepared["label_encoder"], cm_path)

    return {
        "model_path": str(saved_path),
        "evaluation_plot_path": str(plot_path),
        "confusion_matrix_path": str(cm_path),
        "metrics": model_bundle["metrics"],
        "train_size": int(len(x_train)),
        "test_size": int(len(x_test)),
    }


def train_models_from_csv(csv_path: str | Path, target_column: str, output_path: str | Path = "artifacts/hardware_trojan_models.pkl", **kwargs) -> dict:
    return train_three_models(load_csv_data(csv_path), target_column, output_path=output_path, **kwargs)


def train_models_from_huggingface(dataset_id: str, target_column: str, output_path: str | Path = "artifacts/hardware_trojan_models.pkl", *, split: str = "train", revision: str | None = None, **kwargs) -> dict:
    result = train_three_models(load_huggingface_data(dataset_id, split=split, revision=revision), target_column, output_path=output_path, **kwargs)
    result.update({"dataset_id": dataset_id, "split": split, "revision": revision})
    return result


## 7. Leave-One-Trojan-Out Cross-Validation

Trains on all-but-one Trojan variant (plus clean traces), tests on the
held-out variant. This is the validation method that actually demonstrates
the models generalize to an *unseen* Trojan, rather than memorizing traces
from a specific variant that we have already seen.

`variant_column` should hold the fine-grained label (e.g. `trojan_leak`,
`trojan_counter`, `trojan_tiny`, `clean`). Every fold trains on `clean` +
all Trojan variants except one, then tests only on the held-out variant
(mixed with a held-out slice of `clean` traces so both classes are present).

In [ ]:
def leave_one_trojan_out_eval(
    data: pd.DataFrame,
    target_column: str,
    variant_column: str,
    clean_label: str = "clean",
    *,
    trace_column: str | None = None,
    random_state: int = 42,
    cnn_epochs: int = 15,
    cnn_batch_size: int = 32,
    show_epoch_progress: bool = False,
) -> pd.DataFrame:
    """Run leave-one-Trojan-out CV and return a results table.

    For each Trojan variant V:
      - train on: clean rows (80%) + every OTHER Trojan variant's rows
      - test on:  held-out clean rows (20%) + ALL of variant V's rows
    Reports accuracy, weighted F1, and false negative rate (missed Trojans)
    per model per held-out variant.
    """
    variants = sorted(v for v in data[variant_column].unique() if v != clean_label)
    if len(variants) < 2:
        raise ValueError("Need at least 2 Trojan variants for leave-one-out CV.")

    clean_rows = data[data[variant_column] == clean_label]
    clean_train, clean_test = train_test_split(
        clean_rows, test_size=0.2, random_state=random_state
    )

    results = []
    for held_out in variants:
        train_variants = [v for v in variants if v != held_out]
        train_df = pd.concat(
            [clean_train, data[data[variant_column].isin(train_variants)]], ignore_index=True
        )
        test_df = pd.concat(
            [clean_test, data[data[variant_column] == held_out]], ignore_index=True
        )

        x_train_raw, y_train_raw = power_trace_features(train_df, target_column, trace_column)
        x_test_raw, y_test_raw = power_trace_features(test_df, target_column, trace_column)

        scaler = StandardScaler().fit(x_train_raw)
        x_train = scaler.transform(x_train_raw).astype(np.float32)
        x_test = scaler.transform(x_test_raw).astype(np.float32)

        label_encoder = LabelEncoder().fit(y_train_raw)
        y_train = label_encoder.transform(y_train_raw)
        # Map test labels through the same encoder; held-out variant's label
        # may not have appeared in training as its own class if target_column
        # is already binary (clean/trojan) -- that's fine, this still works
        # for a binary clean-vs-trojan target.
        y_test = label_encoder.transform(y_test_raw)

        rf = RandomForestClassifier(
            n_estimators=300, random_state=random_state, n_jobs=-1, class_weight="balanced",
        ).fit(x_train, y_train)
        svm = SVC(
            kernel="rbf", C=1.0, gamma="scale", class_weight="balanced",
            probability=True, random_state=random_state,
        ).fit(x_train, y_train)
        _, cnn_preds, _ = train_1d_cnn(
            x_train, y_train, x_test, y_test,
            epochs=cnn_epochs, batch_size=cnn_batch_size, random_state=random_state,
            show_epoch_progress=show_epoch_progress,
        )

        trojan_class = [c for c in label_encoder.classes_ if c != clean_label]
        for model_name, preds in [
            ("random_forest", rf.predict(x_test)),
            ("svm", svm.predict(x_test)),
            ("cnn_1d", cnn_preds),
        ]:
            trojan_mask = y_test_raw != clean_label
            if trojan_mask.sum() > 0:
                fnr = float((preds[trojan_mask] != y_test[trojan_mask]).mean())
            else:
                fnr = float("nan")
            results.append({
                "held_out_variant": held_out,
                "model": model_name,
                "accuracy": accuracy_score(y_test, preds),
                "f1_weighted": f1_score(y_test, preds, average="weighted", zero_division=0),
                "false_negative_rate": fnr,
                "n_test_trojan_rows": int(trojan_mask.sum()),
            })

    return pd.DataFrame(results)


## 8. Run It

Replace the path/column names with your actual dataset, then run.

In [ ]:
# --- Standard split (quick sanity check) ---
# result = train_models_from_csv(
#     "data/power_traces.csv",
#     target_column="label",          # e.g. clean / trojan
#     output_path="artifacts/hardware_trojan_models.pkl",
# )
# print(result["metrics"])


In [ ]:
# --- Leave-one-Trojan-out (the one to put in your report) ---
# df = load_csv_data("data/power_traces.csv")
# loo_results = leave_one_trojan_out_eval(
#     df,
#     target_column="label",          # binary: clean / trojan
#     variant_column="trojan_variant",# fine-grained: clean / trojan_leak / trojan_counter / trojan_tiny
#     trace_column="power_trace",
# )
# loo_results


## 9. What Else to Add

Beyond leave-one-out CV and the CNN F1 fix already in this notebook, worth
adding before your final report:

- **RF feature importance plot** — `random_forest.feature_importances_` plotted
  against trace sample index. Shows *which part* of the power trace the model
  actually relies on — great for your report and for sanity-checking the model
  isn't keying off noise.
- **Multiple random seeds** — rerun training with 3–5 different `random_state`
  values and report mean ± std of accuracy. Small datasets are noisy; a single
  run's number can be misleading on its own.
- **Inference latency** — time a single prediction per model. Relevant since
  your abstract targets resource-constrained edge/IoT deployment, not just
  detection accuracy.
- **Area/power overhead table** — from Vivado's utilization and power reports,
  comparing clean vs. each Trojan variant's LUT/FF count and estimated power.
  This is a separate axis from ML accuracy but is explicitly one of your
  stated evaluation criteria.
- **ROC curve / AUC** — especially useful if you end up reporting results as
  binary clean-vs-Trojan rather than per-variant multiclass.
- **Cross-check against literature numbers** — plot your accuracy/FNR next to
  the numbers from the papers in your reading list, as a benchmarking table.
